# Practical 4: A Forensic Agent Over an Android Image

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danjethh/sectratek-AI-Lab/blob/main/lab07_llm_android_image_agent/04_lab_notebook.ipynb)

**Course:** AI for Cybersecurity and Digital Forensics  
**Instructor:** Daniel Kwaku Ntiamoah Addai, Mobile Threat Research Lead, SectraTEK Lab  
**Runs in:** Google Colab or any hosted Jupyter. Nothing is installed on your own computer.

> Prefer to run everything **locally** on your own PC with no API key? Use the sister notebook
> `04_lab_notebook_local.ipynb`, which swaps the hosted model for a local Qwen via Ollama.

---

To answer *which accounts are on this phone?* from a device image, an examiner normally has to **know which
artifact to open** and then **write SQL**. In this lab you build a small **agent** that does that
search for you: you ask a plain-English question, and a lightweight **Qwen** model walks the
extraction on its own, reads the right file, and answers **with the exact file it used**.

You build the agent **by hand**:

1. Give the model read-only **tools** (list a folder, read a file, run SQL, find files, search text).
2. Run a **loop**: the model thinks, picks one tool, sees the result, and repeats.
3. Stop when it has an answer, and make it **cite the file**.

## Part 0: One-Time Setup

### Step 0.1: Install the library

In [ ]:
# Purpose: install the one client library this notebook needs, inside the hosted session only.
# openai: a small client for chatting with a model. OpenRouter speaks this same interface,
#         so we point the same client at OpenRouter's free Qwen model.
%pip install -q openai

### Step 0.2: Connect to a free Qwen model (OpenRouter)

1. Open **https://openrouter.ai/keys**, sign in, and create a free key.
2. Run the cell and paste it when asked. It is hidden and is **not saved** into the notebook.

In [ ]:
# Purpose: collect the OpenRouter key privately and build a client for the Qwen model.
import os        # os: lets the notebook read saved settings, such as where your key is kept
import getpass   # getpass: lets you type your secret key without it appearing on screen
from openai import OpenAI  # OpenAI: the client that carries messages to the model

if not os.environ.get('OPENROUTER_API_KEY'):
    os.environ['OPENROUTER_API_KEY'] = getpass.getpass('Paste your OpenRouter API key (hidden): ').strip()

BASE_URL = 'https://openrouter.ai/api/v1'
MODEL = 'qwen/qwen3-30b-a3b:free'  # small, free Qwen. Fallbacks: qwen/qwen3-14b:free, qwen/qwen3-4b:free

client = OpenAI(base_url=BASE_URL, api_key=os.environ['OPENROUTER_API_KEY'])

try:
    probe = client.chat.completions.create(
        model=MODEL, messages=[{'role': 'user', 'content': 'Reply with one word: ready'}])
    print('Model OK:', MODEL, '->', probe.choices[0].message.content.strip()[:40])
except Exception as err:
    print('Could not use', MODEL, '->', err)
    print('\nFree Qwen models to try instead (set MODEL above and re-run):')
    try:
        for m in client.models.list().data:
            if 'qwen' in m.id and ':free' in m.id: print('  -', m.id)
    except Exception as err2:
        print('  (could not list models:', err2, ')')

### Step 0.3: Build the synthetic Android extraction

**Purpose:** a real extraction is gigabytes. We build a small, synthetic stand-in with the same
shape: realistic paths, SQLite databases (Gmail, contacts, SMS, WhatsApp), and some text/XML/JSON
artifacts. Everything in it is fictional.  
**What you'll see:** the files that make up the image.

In [ ]:
# Purpose: create a tiny, synthetic Android device image on disk for the agent to investigate.
import sqlite3   # sqlite3: Python's built-in reader/writer for SQLite files (how Android stores app data)
import os, json
from pathlib import Path  # Path: a readable way to refer to files and folders

ROOT = Path('android_image')
TEXT_FILES = {
    'android_image/build.prop': '# Synthetic build.prop (teaching data, fictional)\nro.product.manufacturer=Google\nro.product.brand=google\nro.product.model=Pixel 7a\nro.product.device=lynx\nro.product.name=lynx\nro.build.version.release=14\nro.build.version.sdk=34\nro.build.fingerprint=google/lynx/lynx:14/AP2A.240705.005/2024072800:user/release-keys\n',
    'android_image/data/system/packages.list': 'com.google.android.gm 10089 0 /data/user/0/com.google.android.gm default 0 1\ncom.android.providers.contacts 10003 0 /data/user/0/com.android.providers.contacts platform 0 1\ncom.android.providers.telephony 10004 0 /data/user/0/com.android.providers.telephony platform 0 1\ncom.whatsapp 10137 0 /data/user/0/com.whatsapp default 0 1\ncom.instagram.android 10140 0 /data/user/0/com.instagram.android default 0 1\norg.thoughtcrime.securesms 10142 0 /data/user/0/org.thoughtcrime.securesms default 0 1\n',
    'android_image/data/system/users/0/settings_secure.xml': '<?xml version=\'1.0\' encoding=\'utf-8\' standalone=\'yes\' ?>\n<settings version="241">\n  <setting id="1" name="android_id" value="a1b2c3d4e5f60718" package="android" />\n  <setting id="2" name="bluetooth_name" value="Pixel 7a" package="com.android.settings" />\n  <setting id="3" name="device_name" value="Nadia\'s Pixel 7a" package="com.android.settings" />\n</settings>\n',
    'android_image/data/misc/wifi/WifiConfigStore.xml': '<?xml version=\'1.0\' encoding=\'utf-8\' standalone=\'yes\' ?>\n<WifiConfigStoreData>\n  <NetworkList>\n    <Network><string name="SSID">&quot;HomeNet_5G&quot;</string><long name="LastConnectedTime" value="1721295600000" /></Network>\n    <Network><string name="SSID">&quot;BrightBeans Cafe&quot;</string><long name="LastConnectedTime" value="1720950000000" /></Network>\n    <Network><string name="SSID">&quot;Okafor-Office&quot;</string><long name="LastConnectedTime" value="1721210400000" /></Network>\n  </NetworkList>\n</WifiConfigStoreData>\n',
    'android_image/data/data/com.whatsapp/files/me.json': '{\n  "jid": "15550143788@s.whatsapp.net",\n  "pushname": "Nadia O",\n  "registered": true,\n  "last_backup": "2024-07-18"\n}',
}
for rel, content in TEXT_FILES.items():
    p = Path(rel); p.parent.mkdir(parents=True, exist_ok=True); p.write_text(content)

def make_db(rel, schema, inserts):
    p = Path(rel); p.parent.mkdir(parents=True, exist_ok=True)
    if p.exists(): p.unlink()
    con = sqlite3.connect(p); con.executescript(schema)
    for sql, params in inserts: con.executemany(sql, params)
    con.commit(); con.close()

make_db('android_image/data/data/com.google.android.gm/databases/accounts.db',
        'CREATE TABLE accounts(name TEXT, type TEXT, last_sync INTEGER);',
        [('INSERT INTO accounts VALUES (?,?,?)', [
            ('nadia.okafor@gmail.com', 'com.google', 1721295600),
            ('+15550143788', 'com.whatsapp', 1721240000)])])
make_db('android_image/data/data/com.android.providers.contacts/databases/contacts2.db',
        'CREATE TABLE contacts(display_name TEXT, phone TEXT, times_contacted INTEGER);',
        [('INSERT INTO contacts VALUES (?,?,?)', [
            ('Marcus Webb', '+15550142210', 31), ('Priya Shah', '+15550148890', 12),
            ('Dad', '+15550149001', 54), ('BrightBeans Cafe', '+15550140077', 3),
            ('Alina K (work)', '+15550147325', 20)])])
make_db('android_image/data/data/com.android.providers.telephony/databases/mmssms.db',
        'CREATE TABLE sms(address TEXT, date INTEGER, body TEXT, type INTEGER);',
        [('INSERT INTO sms VALUES (?,?,?,?)', [
            ('+15550142210', 1721200000000, 'Are we still on for Friday?', 1),
            ('+15550142210', 1721200600000, 'Yes, 6pm at the usual place.', 2),
            ('+15550149001', 1721240500000, 'Landed safe, call you tonight.', 2),
            ('+15550148890', 1721250000000, 'Can you send the invoice?', 1),
            ('+15550149001', 1721299000000, 'Thanks Dad, talk soon.', 2)])])
make_db('android_image/data/data/com.whatsapp/databases/msgstore.db',
        'CREATE TABLE messages(chat TEXT, timestamp INTEGER, from_me INTEGER, text TEXT);',
        [('INSERT INTO messages VALUES (?,?,?,?)', [
            ('Marcus Webb', 1721260000000, 0, 'Did the package arrive?'),
            ('Marcus Webb', 1721260400000, 1, 'Not yet, will check tomorrow.'),
            ('Priya Shah', 1721280000000, 1, 'Sent the files to your email.')])])

print('Built synthetic image under', ROOT.resolve(), '\n')
for dp, dn, fn in sorted(os.walk(ROOT)):
    for f in sorted(fn): print('  ', os.path.join(dp, f))

## Part A: Build the Agent by Hand

An **agent** is a model that can *act*: instead of answering from memory, it calls tools, looks at
what they return, and decides what to do next. You build it in two pieces: the **tools**, then the
**loop**.

### Step A.1: Give the agent its read-only tools

**Purpose:** the agent cannot touch files directly. It can only use the tools we hand it, kept
**read-only** and **locked inside the image folder**. Real extractions hold many file types, so we
give it five tools:

- `list_dir(path)` : list what is inside a folder.
- `read_file(path)` : show a file's text (pretty-prints JSON; points you to `run_sql` for databases).
- `run_sql(path, query)` : run one read-only SQL SELECT on a database.
- `find_files(name)` : locate files anywhere in the image by name.
- `search_text(keyword)` : search inside text files for a word, to find which file to open next.

**What you'll see:** a quick self-test of the tools against the image.

In [ ]:
# Purpose: define the small, safe toolbox the agent may use. Nothing here can change the image.
import sqlite3, os, json
from pathlib import Path

TEXT_EXT = {'.txt', '.xml', '.json', '.prop', '.list', '.log', '.csv', '.ini', '.conf'}

def _resolve(path):
    # Turn a path from the model into a real path, and refuse anything outside the image folder.
    base = ROOT.resolve()
    path = (path or '.').strip().lstrip('/')
    target = base if path in ('', '.') else (ROOT / path).resolve()
    if target != base and base not in target.parents:
        raise ValueError('path is outside the image')
    return target

def list_dir(path='.'):
    # List the entries in a folder (folders shown with a trailing '/').
    t = _resolve(path)
    if not t.exists():  return f'No such path: {path}'
    if t.is_file():     return f'{path} is a file, not a folder. Use read_file or run_sql.'
    items = [e + ('/' if (t / e).is_dir() else '') for e in sorted(os.listdir(t))]
    return '\n'.join(items) if items else '(empty folder)'

def read_file(path, max_chars=1500):
    # Show a text file. Pretty-prints JSON; for a SQLite database, points you to run_sql.
    t = _resolve(path)
    if not t.exists() or t.is_dir():  return f'No such file: {path}'
    head = t.read_bytes()[:16]
    if head[:15] == b'SQLite format 3':
        con = sqlite3.connect(f'file:{t}?mode=ro', uri=True)
        tables = [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")]
        con.close()
        return f'[{path} is a SQLite database. Tables: {tables}. Use run_sql with a SELECT query.]'
    if t.suffix.lower() not in TEXT_EXT and b'\x00' in head:
        return f'[{path} is a binary file ({t.stat().st_size} bytes). First bytes (hex): {head.hex(" ")}.]'
    text = t.read_text(errors='replace')
    if t.suffix.lower() == '.json':
        try: text = json.dumps(json.loads(text), indent=2)
        except Exception: pass
    return text[:max_chars] + ('' if len(text) <= max_chars else '\n...[truncated]')

def run_sql(path, query):
    # Run ONE read-only SELECT against a SQLite database and return the rows.
    t = _resolve(path)
    if not t.exists():  return f'No such database: {path}'
    q = (query or '').strip().rstrip(';')
    if not q.lower().startswith(('select', 'pragma')):
        return 'Only read-only SELECT queries are allowed.'
    try:
        con = sqlite3.connect(f'file:{t}?mode=ro', uri=True)  # mode=ro: read-only, cannot change evidence
        cur = con.execute(q); cols = [c[0] for c in cur.description] if cur.description else []
        rows = cur.fetchmany(50); con.close()
    except Exception as e:
        return f'SQL error: {e}'
    if not rows: return '(no rows)'
    return '\n'.join([' | '.join(cols)] + [' | '.join(str(c) for c in r) for r in rows])

def find_files(name):
    # Find files anywhere in the image whose path contains `name` (case-insensitive).
    base = ROOT.resolve()
    hits = [str(p.relative_to(base)) for p in sorted(base.rglob('*'))
            if p.is_file() and (name or '').lower() in str(p.relative_to(base)).lower()]
    return '\n'.join(hits[:50]) if hits else f'No files matched {name!r}.'

def search_text(keyword, max_hits=20):
    # Search inside readable text files for a keyword; returns path:line for each match.
    base = ROOT.resolve(); out = []
    for p in sorted(base.rglob('*')):
        if not p.is_file() or p.suffix.lower() not in TEXT_EXT: continue
        for i, line in enumerate(p.read_text(errors='replace').splitlines(), 1):
            if (keyword or '').lower() in line.lower():
                out.append(f'{p.relative_to(base)}:{i}: {line.strip()[:120]}')
                if len(out) >= max_hits: return '\n'.join(out)
    return '\n'.join(out) if out else f'No text matches for {keyword!r}.'

TOOLS = {'list_dir': list_dir, 'read_file': read_file, 'run_sql': run_sql,
         'find_files': find_files, 'search_text': search_text}

print('list_dir("data/data") ->\n' + list_dir('data/data'))
print('\nfind_files("accounts") ->\n' + find_files('accounts'))
print('\nsearch_text("android_id") ->\n' + search_text('android_id'))

### Step A.2: Teach the agent how to speak, and give it its case context

**Purpose:** the model needs (1) its **job** (it is doing digital forensics) and (2) a **format** to
reply in, so our code can read its choices. The system prompt below gives it both: the case context,
the tool list, and a rule to reply with one JSON object per step.  
**What you'll see:** nothing yet; this defines the rulebook and a safe JSON reader.

In [ ]:
# Purpose: the case context + rulebook the model follows, plus a tolerant reader for its JSON.
import json, re

SYSTEM_PROMPT = '''You are Sectra, a digital-forensics examiner's assistant. You are investigating an
Android device extraction (a copy of a phone's files) to answer the examiner's question. You cannot
see any file until you open it with a tool, and every answer must be based ONLY on what the tools return.

TOOLS (use exactly one per step):
- list_dir(path): list the entries in a folder. Use "." for the top of the image.
- read_file(path): show the text contents of a file.
- run_sql(path, query): run ONE read-only SQL SELECT against a SQLite database and return rows.
- find_files(name): list files anywhere in the image whose path contains name.
- search_text(keyword): search inside text files for a keyword; returns path:line matches.

Reply with ONE JSON object and nothing else.
To use a tool:
  {"thought": "<one short sentence>", "action": "list_dir", "action_input": {"path": "data/data"}}
When you can answer:
  {"thought": "<one short sentence>", "final_answer": "<your answer>", "evidence_path": "<the file you used>"}

RULES: investigate step by step; use ONLY what the tools return; never invent paths, tables, or data;
if a tool errors, try another path or tool; if the evidence is not there, say so plainly; always cite
the exact file in evidence_path. Keep thoughts to one sentence. /no_think'''

def parse_action(raw):
    # Pull the JSON object out of the model's reply, tolerating <think> blocks and ``` code fences.
    text = re.sub(r'<think>.*?</think>', '', raw, flags=re.S)
    text = text.replace('```json', '').replace('```', '')
    start = text.find('{')
    if start == -1: return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == '{': depth += 1
        elif text[i] == '}':
            depth -= 1
            if depth == 0:
                try: return json.loads(text[start:i + 1])
                except Exception: return None
    return None

### Step A.3: Run the agent loop

**Purpose:** the whole agent: ask the model for its next move, run that one tool, feed the result
back, repeat, until it answers. This think-act-observe cycle is the **ReAct** pattern.  
**What you'll see:** the reasoning trace, then the final cited answer.

In [ ]:
# Purpose: the agent loop. It drives the model think -> act -> observe until it can answer.
def run_agent(question, max_steps=12, verbose=True):
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': f'Investigate and answer this question: {question}'}]
    for step in range(1, max_steps + 1):
        raw = client.chat.completions.create(
            model=MODEL, messages=messages, temperature=0).choices[0].message.content
        messages.append({'role': 'assistant', 'content': raw})
        move = parse_action(raw)
        if move is None:
            messages.append({'role': 'user', 'content': 'That was not valid JSON. Reply with ONE JSON object only.'})
            continue
        if verbose and move.get('thought'): print(f'Step {step} | thought: {move["thought"]}')
        if 'final_answer' in move:
            if verbose:
                print(f'\nANSWER: {move["final_answer"]}')
                print(f'EVIDENCE: {move.get("evidence_path", "(none given)")}')
            return move
        tool, args = move.get('action'), move.get('action_input', {}) or {}
        if tool not in TOOLS:
            observation = f'Unknown tool {tool!r}. Use one of: ' + ', '.join(TOOLS)
        else:
            try: observation = TOOLS[tool](**args)
            except Exception as e: observation = f'Tool error: {e}'
        if verbose:
            short = observation if len(observation) < 300 else observation[:300] + ' ...'
            print(f'        action: {tool}({args})')
            print(f'        result: {short}\n')
        messages.append({'role': 'user', 'content': f'Observation:\n{observation}'})
    print('Stopped: reached the step limit without a final answer.')
    return None

result = run_agent('Which email accounts are signed in on this device?')

## Part B: Why the Agent Matters

Ask the **same question two ways**: once to the Qwen model alone (no tools, no image), and once
through the agent. The plain model has no access to this device, so it can only guess.

### Step B.1: Ask the model with no tools

In [ ]:
bare = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': 'Which email accounts are signed in on this device? Answer directly.'}],
).choices[0].message.content
print(bare)

### Step B.2: Ask the same question through the agent

In [ ]:
result = run_agent('Which email accounts are signed in on this device?')

## Part C: Ask Your Own Investigation Questions

Use `run_agent(...)` to ask the image your own questions. Try ones that need the new tools too, like
searching for a keyword or finding a file by name.

In [ ]:
result = run_agent('What phone model is this and which Android version does it run?')

### Good questions to try

- *How many contacts are saved on this device?*
- *What is the most recent text message that was sent, and to which number?*
- *Which messaging or social apps are installed?*  (hint: the agent can `search_text` or read `packages.list`)
- *What Wi-Fi networks has this device connected to?*
- *Find any file that mentions WhatsApp, then tell me the account's push name.*

If the agent takes a wrong turn or runs out of steps, re-run the cell, or name the kind of artifact
you expect (an account, a message, an app, a Wi-Fi network).

## Run Sectra on a REAL extraction (optional)

The image above is synthetic, so the lab runs instantly for everyone. But Sectra explores whatever
tree you point it at, so you can aim it at a **real** mobile extraction too.

**One honest caveat:** a real *full file system* (FFS) extraction is big. The public, no-login,
teaching-standard image below (a Google Pixel 3 running Android 13, published by Josh Hickman and
mirrored by **Digital Corpora**) is about **18.5 GB to download and ~40 GB once extracted**. That is
fine on a local machine or a high-disk Colab runtime, but it is a real download - so the cell that
fetches it is left **commented out**, and nothing runs until you choose to. Your three options:

1. **Synthetic** (default) - what you already used; instant.
2. **Real image** - run the fetch below (big; best run locally or on a roomy runtime).
3. **Bring your own** - already have an extraction folder? `use_extraction('/path/to/it')`.

Other good public sources: Digital Corpora's other mobile images (`digitalcorpora.org`), and Josh
Hickman's image index at `thebinaryhick.blog/public_images`.

In [ ]:
import shutil, tarfile, urllib.request

REAL_SOURCES = {
    # Josh Hickman's Pixel 3 / Android 13 FFS image, mirrored by Digital Corpora (academic, no login).
    'digitalcorpora_android13': {
        'url': 'https://digitalcorpora.s3.amazonaws.com/corpora/mobile/android_13/android_13_data.tar.gz',
        'download_gb': 18.5, 'extracted_gb': 40,
    },
}

def _download(url, dest):
    last = [0]
    def hook(block, bs, total):
        done = block * bs
        if total > 0 and done - last[0] > 250 * 1024 * 1024:   # report every ~250 MB
            last[0] = done; print(f'  {done/1e9:5.1f} / {total/1e9:.1f} GB  ({100*done/total:3.0f}%)')
    urllib.request.urlretrieve(url, dest, reporthook=hook)

def fetch_extraction(source='digitalcorpora_android13', dest='real_extraction', max_gb=60, force=False):
    """Download + extract a real public FFS image. Large - read the printed size first."""
    s = REAL_SOURCES[source]
    need = s['download_gb'] + s['extracted_gb']
    free = shutil.disk_usage('.').free / 1e9
    print(f"{source}: ~{s['download_gb']} GB download, ~{s['extracted_gb']} GB extracted.")
    print(f'Free disk here: {free:.0f} GB.  Needed (rough): {need:.0f} GB.')
    if not force and free < need:
        raise SystemExit('Not enough free disk. Use the synthetic image, bring your own, or run locally. '
                         '(force=True to try anyway.)')
    if not force and s['download_gb'] > max_gb:
        raise SystemExit(f"This image is ~{s['download_gb']} GB (> max_gb={max_gb}). Raise max_gb or force=True.")
    dest = Path(dest); dest.mkdir(exist_ok=True)
    tgz = dest / 'image.tar.gz'
    if not tgz.exists() or tgz.stat().st_size == 0:
        print('Downloading (this takes a while)...'); _download(s['url'], tgz)
    print('Extracting...')
    try:
        with tarfile.open(tgz) as t: t.extractall(dest, filter='data')   # Python 3.12+ safe filter
    except TypeError:
        with tarfile.open(tgz) as t: t.extractall(dest)
    print('Done. Extracted under', dest.resolve())
    return dest

def use_extraction(path='real_extraction'):
    """Point Sectra at an extraction folder: rebinds ROOT, so your run_agent(...) cells now walk it."""
    global ROOT
    ROOT = Path(path)
    print('ROOT is now', ROOT.resolve())
    try:
        print('Top-level entries:', sorted(os.listdir(ROOT))[:20])
    except Exception as e:
        print('(could not list it yet:', e, ')')
    print('Re-run your run_agent(...) cells - Sectra now investigates this extraction.')
    return ROOT

print('Ready. Uncomment the next cell to fetch a real image, or call use_extraction(\'/your/path\').')

Uncomment to fetch and use the real image. On a real FFS tree, prefer `list_dir`, `find_files`, and
`run_sql` over `search_text` (which scans every file), and ask targeted questions.

In [ ]:
# root = fetch_extraction('digitalcorpora_android13')   # ~18.5 GB download - only when you mean it
# use_extraction(root)
# run_agent('Which messaging apps are installed, and what is the device model?')

## Part D (Optional, Advanced): Let the Agent Write Code

So far the agent can only call the five fixed tools. You can also give it a tool that runs **Python
it writes itself**, so it can handle anything the fixed tools do not cover. This is powerful but
risky: the model's text becomes code that runs on your machine. We keep it **off by default**, block
obvious writes, and run it in a small namespace. Treat it as a demonstration of the trade-off, not
something to point at real evidence unsupervised.

In [ ]:
# Purpose (ADVANCED, OPTIONAL): add a tool that executes a short, read-only Python snippet.
import io, contextlib

def run_python(code):
    # ROOT (the image folder) is available to the snippet. Writing to files is blocked.
    if any(b in code for b in ('open(', 'write', 'unlink', 'remove', 'rmtree', 'os.system',
                               'subprocess', 'Popen', '__import__')):
        return 'Blocked: that snippet looks like it writes or runs external commands.'
    buf = io.StringIO(); safe = {'ROOT': ROOT, 'sqlite3': sqlite3, 'json': json, 'Path': Path, 're': re}
    try:
        with contextlib.redirect_stdout(buf): exec(code, safe)
    except Exception as e:
        return f'Python error: {e}'
    out = buf.getvalue().strip()
    return out[:1500] if out else '(ran, but printed nothing)'

# Enable it by adding it to the toolbox, then mention it in the system prompt if you want the agent to use it:
TOOLS['run_python'] = run_python
print(run_python("print(sorted(p.name for p in ROOT.rglob('*.db')))"))

## Your Short Write-Up

1. Pick one question. Paste the agent's answer **and** the evidence path, then check the file.
2. Compare Part B.1 and B.2: what did the plain model do that the agent did not?
3. In one sentence: why does making the agent **cite the file** make its answer safer to rely on?

In [ ]:
writeup = {
    'q1_answer_and_evidence': '...',
    'q2_plain_vs_agent': '...',
    'q3_why_citing_helps': '...',
}
for k, v in writeup.items(): print(k, '->', v)

## Submit

Save the notebook with the reasoning trace, the Part B comparison, and your write-up.

---

*The device image is synthetic. The agent (tools + the think-act-observe loop) is built from scratch, driven by a small open-source Qwen model through OpenRouter, so you can see every step and the exact file behind every answer.*